In [89]:
import os
import re
import json
import requests
import pandas as pd
from pathlib import Path

# =========================
# Config
# =========================
DOWNLOAD_DIR = Path("gitbugs_downloads")
CLEANED_DIR = Path("gitbugs_cleaned")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Helpers
# =========================
def normalize_col(col: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(col).strip().lower())

def detect_issue_id_col(columns):
    normalized = {col: normalize_col(col) for col in columns}
    preferred = [
        "issueid",
        "bugid",
        "id",
        "issuekey",
        "bugnumber",
        "issuenumber",
    ]

    for target in preferred:
        for col, norm in normalized.items():
            if norm == target:
                return col

    for col, norm in normalized.items():
        if "issue" in norm and "id" in norm:
            return col
    for col, norm in normalized.items():
        if "bug" in norm and "id" in norm:
            return col

    raise ValueError(f"Could not detect issue-id column from columns: {list(columns)}")

def detect_combined_cols(columns):
    normalized = {col: normalize_col(col) for col in columns}

    issue_col = None
    dup_col = None

    for col, norm in normalized.items():
        if norm in {"issueid", "bugid", "id"} and issue_col is None:
            issue_col = col
        if norm in {"duplicateid", "dupid", "duplicatedbugid"} and dup_col is None:
            dup_col = col

    if issue_col is None:
        for col, norm in normalized.items():
            if "issue" in norm and "id" in norm:
                issue_col = col
                break
    if dup_col is None:
        for col, norm in normalized.items():
            if "duplicate" in norm and "id" in norm:
                dup_col = col
                break

    if issue_col is None or dup_col is None:
        cols = list(columns)
        if len(cols) >= 2:
            issue_col = issue_col or cols[0]
            dup_col = dup_col or cols[1]
        else:
            raise ValueError(f"Could not detect combined-file columns from columns: {list(columns)}")

    return issue_col, dup_col

def split_ids(value):
    if pd.isna(value):
        return []

    s = str(value).strip()
    if not s:
        return []

    parts = re.split(r"[,;|]+", s)
    cleaned = []
    for p in parts:
        p = p.strip().strip('"').strip("'")
        if p:
            cleaned.append(p)
    return cleaned

def is_bug_file(path: Path) -> bool:
    """
    Local bug file:
      *_bugs.csv
    but not:
      *combined.csv
      *-train.csv
      *-test.csv
      already-cleaned outputs
    """
    name = path.name.lower()

    if not name.endswith(".csv"):
        return False

    if "combined" in name:
        return False
    if "-train" in name or "-test" in name:
        return False
    if "_minus_combined" in name:
        return False

    return "_bugs" in name

def get_bug_prefix(filename: str) -> str:
    """
    cassandra_bugs.csv -> cassandra
    Core_bugs.csv -> core
    """
    name = filename.lower()
    if name.endswith(".csv"):
        name = name[:-4]

    idx = name.find("_bugs")
    if idx != -1:
        return name[:idx]

    return name

def find_local_pairs(download_dir: Path):
    """
    Scan gitbugs_downloads locally and pair:
      <something>_bugs.csv
      <something>_bugs-combined.csv
    inside each project folder.
    """
    pairs = []

    for project_dir in sorted(download_dir.iterdir()):
        if not project_dir.is_dir():
            continue

        csv_files = list(project_dir.glob("*.csv"))

        bug_files = [f for f in csv_files if is_bug_file(f)]
        combined_files = [f for f in csv_files if "combined" in f.name.lower()]

        if not bug_files:
            print(f"[WARN] No bug file found in {project_dir}")
            continue

        if not combined_files:
            print(f"[WARN] No combined file found in {project_dir}")
            continue

        for bug_file in bug_files:
            bug_prefix = get_bug_prefix(bug_file.name)
            chosen = None

            for c in combined_files:
                c_name = c.name.lower()
                if bug_prefix in c_name:
                    chosen = c
                    break

            if chosen is None:
                chosen = combined_files[0]

            pairs.append((bug_file, chosen))

    return pairs

def subtract_combined_from_bugs(bugs_csv_path: Path, combined_csv_path: Path):
    bugs_df = pd.read_csv(bugs_csv_path, dtype=str, low_memory=False)
    combined_df = pd.read_csv(combined_csv_path, dtype=str, low_memory=False)

    bug_id_col = detect_issue_id_col(bugs_df.columns)
    combined_issue_col, combined_dup_col = detect_combined_cols(combined_df.columns)

    bug_ids_to_remove = set()

    for v in combined_df[combined_issue_col]:
        bug_ids_to_remove.update(split_ids(v))

    for v in combined_df[combined_dup_col]:
        bug_ids_to_remove.update(split_ids(v))

    before_n = len(bugs_df)

    bug_ids_series = bugs_df[bug_id_col].astype(str).str.strip()
    cleaned_df = bugs_df.loc[~bug_ids_series.isin(bug_ids_to_remove)].copy()

    after_n = len(cleaned_df)
    removed_n = before_n - after_n

    return cleaned_df, {
        "bug_id_col": bug_id_col,
        "combined_issue_col": combined_issue_col,
        "combined_dup_col": combined_dup_col,
        "before_rows": before_n,
        "after_rows": after_n,
        "removed_rows": removed_n,
        "unique_ids_in_combined": len(bug_ids_to_remove),
    }

# =========================
# Main
# =========================

print(f"Scanning local files in: {DOWNLOAD_DIR.resolve()}")
pairs = find_local_pairs(DOWNLOAD_DIR)

print(f"Matched local bug/combined pairs: {len(pairs)}\n")

all_cleaned = []

for bug_local, combined_local in pairs:
    print("=" * 80)
    print(f"Using bug file      : {bug_local}")
    print(f"Using combined file : {combined_local}")

    cleaned_df, stats = subtract_combined_from_bugs(bug_local, combined_local)

    project_name = bug_local.parent.name
    cleaned_df["source_project"] = project_name

    out_file = CLEANED_DIR / f"{project_name}_bugs_minus_combined.csv"
    cleaned_df.to_csv(out_file, index=False)

    print(f"Project: {project_name}")
    print(f"Detected bug id column            : {stats['bug_id_col']}")
    print(f"Detected combined issue id col    : {stats['combined_issue_col']}")
    print(f"Detected combined duplicate id col: {stats['combined_dup_col']}")
    print(f"Original bug rows                 : {stats['before_rows']}")
    print(f"Unique IDs gathered from combined : {stats['unique_ids_in_combined']}")
    print(f"Rows removed after subtraction    : {stats['removed_rows']}")
    print(f"Rows left after subtraction       : {stats['after_rows']}")
    print(f"Saved cleaned file                : {out_file}\n")

    print(cleaned_df.head())
    all_cleaned.append(cleaned_df)

if not all_cleaned:
    print("No cleaned dataframes were created.")


merged_df = pd.concat(all_cleaned, ignore_index=True)

merged_df = merged_df[merged_df['Priority'].notna() & (merged_df['Priority'] != '--')]
merged_df= merged_df[merged_df['Resolution'].notna()]

merged_path = CLEANED_DIR / "all_projects_bugs_minus_combined.csv"
merged_df.to_csv(merged_path, index=False)

merged_df = merged_df.rename(columns={
    "Summary": "title",
    "Issue id": "id",
    "Priority": "priority",
    "Resolution": "resolution",
    "Created": "created_at",
    "Resolved": "closed_at",
    "Description": "body",
    "source_project": "project",
})

merged_df.drop(columns=['Status', 'Affects Version/s'], inplace=True, errors='ignore')

merged_df.to_csv("/Users/anshikabajpai/Desktop/github/Auto_triage_detection_assistant/data/raw/gitbugs.csv", index=False)

print("=" * 80)
print("FINAL MERGE")
print(f"Number of cleaned project dataframes merged: {len(all_cleaned)}")
print(f"Final merged dataframe size: {merged_df.shape}")
print(f"Saved merged dataframe to : {merged_path}")



Scanning local files in: /Users/anshikabajpai/Desktop/github/Auto_triage_detection_assistant/notebooks/gitbugs_downloads
Matched local bug/combined pairs: 9

Using bug file      : gitbugs_downloads/cassandra/cassandra_bugs.csv
Using combined file : gitbugs_downloads/cassandra/cassandra_bugs-combined.csv
Project: cassandra
Detected bug id column            : Issue id
Detected combined issue id col    : Issue id
Detected combined duplicate id col: Duplicate id
Original bug rows                 : 4612
Unique IDs gathered from combined : 309
Rows removed after subtraction    : 309
Rows left after subtraction       : 4303
Saved cleaned file                : gitbugs_cleaned/cassandra_bugs_minus_combined.csv

                                                               Summary  \
0                    Upgrade to logback 1.2.3 to address CVE-2017-5929   
1                                 Upgrade Cassandra Java Driver to 4.6   
2  provide a configuration option such as endpoint_verification_me

In [35]:
merged_df = merged_df[merged_df['Priority'].notna() & (merged_df['Priority'] != '--')]

In [36]:
merged_df['Priority'].unique()

<ArrowStringArray>
[    'High',   'Normal',      'Low',   'Urgent',       'P3',       'P5',
       'P2',       'P1',       'P4',  'Blocker', 'Critical',    'Major',
    'Minor',  'Trivial']
Length: 14, dtype: str

In [38]:
merged_df['Resolution'].unique()

<ArrowStringArray>
[           'Duplicate',                'Fixed',                    nan,
            'Won't Fix',             'Resolved',        'Not A Problem',
          'Auto Closed',                'Later',     'Cannot Reproduce',
              'Invalid', 'Information Provided',            'Not A Bug',
         'Works for Me',            'Abandoned',                 'Done',
             'Won't Do',          'Implemented',           'Workaround',
           'Incomplete',           'INCOMPLETE',                'FIXED',
                'MOVED',           'WORKSFORME',              'WONTFIX',
              'INVALID',             'INACTIVE',       'Pending Closed',
    'Feedback Received',            'Delivered']
Length: 29, dtype: str

In [57]:
merged_df[merged_df['Resolution'].isna() &  merged_df['Resolved'].notna()]["source_project"].value_counts()

source_project
mozillacore    8649
firefox        2758
thunderbird     310
seamonkey        78
Name: count, dtype: int64

In [63]:
tab=merged_df[merged_df['Resolution'].isna() &  merged_df['Resolved'].notna()]
tab[tab['source_project']=='seamonkey']

,Summary,Issue id,Status,Priority,Resolution,Created,Resolved,Description,source_project,Affects Version/s
129639,"Release Notes: Drop ""Features"" link to Wiki",1607002,NEW,P5,NaN,2020-01-04 12:52:00+00:00,2020-03-08 07:36:18+00:00,AFAIK there has never been useful contents on ...,seamonkey,NaN
129698,Picture truncated at the top because shifted u...,1629489,UNCONFIRMED,P3,NaN,2020-04-13 08:35:30+00:00,2023-07-08 19:50:05+00:00,Steps how to reproduce with installation of un...,seamonkey,NaN
129714,"""downloadmanager.properties"": Links to Microso...",1635010,NEW,P5,NaN,2020-05-04 05:05:58+00:00,2020-05-04 13:34:53+00:00,"File ""downloadmanager.properties"" from seamonk...",seamonkey,NaN
129718,"De Help ""Composer-Tastenkombinationen"": wrong ...",1638419,UNCONFIRMED,P4,NaN,2020-05-15 18:11:34+00:00,2023-07-14 06:10:27+00:00,Help Tells: \n\n| Befehl | Window...,seamonkey,NaN
129723,[De] mailnews_offline.xhtml: Typos,1639101,ASSIGNED,P5,NaN,2020-05-19 05:22:14+00:00,2023-09-25 10:39:07+00:00,Steps how to reproduce NOT reproducible REPROD...,seamonkey,NaN
...,...,...,...,...,...,...,...,...,...,...
130466,www.aok.de - Unable to dismiss the cookie bann...,1909155,NEW,P3,NaN,2024-07-22 06:41:25+00:00,2024-09-04 19:30:20+00:00,From github: https://github.com/webcompat/web-...,seamonkey,NaN
130470,id.bund.de - Blank page,1910827,NEW,P3,NaN,2024-07-31 06:20:25+00:00,2024-09-04 19:26:22+00:00,From github: https://github.com/webcompat/web-...,seamonkey,NaN
130471,"avm.de - ""Download"" buttons do not work",1910828,NEW,P3,NaN,2024-07-31 06:26:48+00:00,2024-09-04 19:39:17+00:00,From github: https://github.com/webcompat/web-...,seamonkey,NaN
130478,volksfreund.de - The buttons are not displayed,1912390,NEW,P3,NaN,2024-08-09 07:55:50+00:00,2024-09-04 19:22:50+00:00,**Environment:**\nOperating System: Windows 10...,seamonkey,NaN


In [ ]:
merged_df= merged_df[merged_df['Resolution'].notna()]

False

In [ ]:
import pandas as pd

# -----------------------------
# 1. Copy dataframe
# -----------------------------
df = merged_df.copy()

# -----------------------------
# 2. Priority -> triage mapping
# -----------------------------
triage_mapping = {
    "High": "P1",
    "Normal": "P3",
    "Low": "P4",
    "Urgent": "P0",
    "P1": "P0",
    "P2": "P1",
    "P3": "P2",
    "P4": "P3",
    "P5": "P4",
    "Blocker": "P0",
    "Critical": "P1",
    "Major": "P2",
    "Minor": "P3",
    "Trivial": "P4",
}

# -----------------------------
# 3. Non-fix resolutions
# -----------------------------
WONTFIX_RESOLUTIONS = {
    # "DUPLICATE",
    "WON'T FIX",
    "WONTFIX",
    "WON'T DO",
    "NOT A PROBLEM",
    "CANNOT REPRODUCE",
    "INVALID",
    "NOT A BUG",
    "WORKS FOR ME",
    "WORKSFORME",
    "INCOMPLETE",
    "ABANDONED",
    "MOVED",
    "INACTIVE",
    "AUTO CLOSED",
    "PENDING CLOSED",
}

# -----------------------------
# 4. Normalize columns
# -----------------------------
df["priority_mapped"] = (
    df["Priority"]
    .astype("string")
    .str.strip()
    .map(triage_mapping)
)

df["resolution_clean"] = (
    df["Resolution"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# -----------------------------
# 5. Parse datetimes safely
# -----------------------------
df["Created"] = pd.to_datetime(df["Created"], errors="coerce", utc=True)
df["Resolved"] = pd.to_datetime(df["Resolved"], errors="coerce", utc=True)

# -----------------------------
# 6. Resolution time in days
# -----------------------------
df["resolution_time_days"] = (
    (df["Resolved"] - df["Created"]).dt.total_seconds() / 86400
)

# -----------------------------
# 7. Condition-wise validation
# -----------------------------
df["cond1_priority_high"] = df["priority_mapped"].isin(["P0", "P1"])
df["cond2_nonfix_resolution"] = df["resolution_clean"].isin(WONTFIX_RESOLUTIONS)
df["cond3_time_gt_7"] = df["resolution_time_days"] > 7

df["is_over_escalated"] = (
    df["cond1_priority_high"] &
    df["cond2_nonfix_resolution"] &
    df["cond3_time_gt_7"]
)

# -----------------------------
# 8. Function version too
# -----------------------------
def _is_over_escalated_gitbugs(row: pd.Series) -> bool:
    cond1 = row["priority_mapped"] in ("P0", "P1")
    cond2 = pd.notna(row["resolution_clean"]) and row["resolution_clean"] in WONTFIX_RESOLUTIONS
    cond3 = pd.notna(row["resolution_time_days"]) and row["resolution_time_days"] > 7
    return cond1 and cond2 and cond3

# optional cross-check
df["is_over_escalated_func"] = df.apply(_is_over_escalated_gitbugs, axis=1)

# -----------------------------
# 9. View matching rows
# -----------------------------
over_escalated_rows = df[df["is_over_escalated"]]

print("Total rows:", len(df))
print("Cond1 priority high:", df["cond1_priority_high"].sum())
print("Cond2 non-fix resolution:", df["cond2_nonfix_resolution"].sum())
print("Cond3 resolution time > 7:", df["cond3_time_gt_7"].sum())
print("Over-escalated rows:", df["is_over_escalated"].sum())

display(
    over_escalated_rows[
        [
            "Priority",
            "priority_mapped",
            "Resolution",
            "resolution_clean",
            "Created",
            "Resolved",
            # "resolution_time_days",
            # "cond1_priority_high",
            # "cond2_nonfix_resolution",
            # "cond3_time_gt_7",
            # "is_over_escalated",
            "Description",
        ]
    ].head(50)
)

/var/folders/bc/g2l1lwnx4cg8sszchc8lfgs00000gn/T/ipykernel_1074/25319430.py:70: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Created"] = pd.to_datetime(df["Created"], errors="coerce", utc=True)
/var/folders/bc/g2l1lwnx4cg8sszchc8lfgs00000gn/T/ipykernel_1074/25319430.py:71: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Resolved"] = pd.to_datetime(df["Resolved"], errors="coerce", utc=True)


Total rows: 85117
Cond1 priority high: 12520
Cond2 non-fix resolution: 28279
Cond3 resolution time > 7: 56157
Over-escalated rows: 1091


,Priority,priority_mapped,Resolution,resolution_clean,Created,Resolved,Description
0,High,P1,Duplicate,DUPLICATE,2020-05-21 22:58:00+00:00,2021-09-23 14:02:00+00:00,Recent scan results identified the following C...
1029,Urgent,P0,Invalid,INVALID,2022-12-08 16:28:00+00:00,2022-12-21 07:35:00+00:00,Our application uses Casandra 3.11.x and has ...
1642,High,P1,Won't Fix,WON'T FIX,2022-02-07 17:36:00+00:00,2024-05-08 08:18:00+00:00,"We depend on java-driver for testing, and deve..."
1643,High,P1,Duplicate,DUPLICATE,2022-03-30 17:20:00+00:00,2022-04-27 13:17:00+00:00,"Testing rebuild in C* 4.0.1, I ran into a scen..."
2508,Urgent,P0,Duplicate,DUPLICATE,2021-07-13 11:42:00+00:00,2021-08-06 15:54:00+00:00,Trying to start 4.0.0 on aarch64 with Java8 th...
3425,Urgent,P0,Invalid,INVALID,2023-12-20 10:58:00+00:00,2024-01-31 12:44:00+00:00,On Cassandra 4.0.7 we are seeing Merkle tree r...
4099,High,P1,Duplicate,DUPLICATE,2021-06-14 14:52:00+00:00,2023-07-28 11:19:00+00:00,A JAR dependency is flagged in Cassandra 3.11....
4315,P2,P1,MOVED,MOVED,2020-01-03 09:06:53+00:00,2020-09-10 01:51:03+00:00,User Agent: Mozilla/5.0 (Windows NT 10.0; Win6...
4323,P2,P1,WONTFIX,WONTFIX,2020-01-04 15:17:13+00:00,2021-04-05 15:22:23+00:00,User Agent: Mozilla/5.0 (Windows NT 10.0; Win6...
4358,P2,P1,WONTFIX,WONTFIX,2020-01-08 15:41:09+00:00,2020-01-22 22:54:33+00:00,"When Search Tips land, we'll have two restrict..."


In [80]:
pd.set_option('display.max_colwidth', None)

over_escalated_rows['Description'][6443]



'User Agent: Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:78.0) Gecko/20100101 Firefox/78.0\n\nSteps to reproduce:\n\nWhile using Firefox 78.0.1 (64-bit) on Ubuntu 18.04.4:  Selected Preferences --> Home --> Top Sites \n\n\nActual results:\n\nBlank new tab page with nothing showing\n\n\nExpected results:\n\nNew tab page should have showed the usual icons of Top Sites, Highlights, Snippets'

In [ ]:
pd.reset_option('display.max_colwidth')

In [ ]:
merged_df = merged_df.rename(columns={
    "title": "Summary",
    "id": "Issue id",
    "priority": "Priority",
    "resolution": "Resolution",
    "created_at": "Created",
    "closed_at": "Resolved",
    "body": "Description",
    "project": "source_project",
})

merged_df.drop(columns=['Status', 'Affects Version/s'], inplace=True, errors='ignore')

In [85]:
merged_df.drop(columns=['Status', 'Affects Version/s'], inplace=True, errors='ignore')

In [86]:
merged_df

,Summary,Issue id,Priority,Resolution,Created,Resolved,Description,source_project
0,Upgrade to logback 1.2.3 to address CVE-2017-5929,13306629,High,Duplicate,21/May/20 22:58,23/Sep/21 14:02,Recent scan results identified the following CVE that requires this upgrade to address\r\n\r\n[https://nvd.nist.gov/vuln/detail/CVE-2017-5929],cassandra
1,Upgrade Cassandra Java Driver to 4.6,13300235,Normal,Fixed,22/Apr/20 10:11,04/Jun/24 08:20,After releasing C* 4.0 I think it would be a good idea to upgrade to the latest Cassandra Java Driver that is being used internally.,cassandra
3,DOC - Primary range repair should be run on every datacenter,13298084,Normal,Fixed,13/Apr/20 17:28,16/Feb/24 12:43,"In the Cassandra document page: [http://cassandra.apache.org/doc/latest/operating/repair.html], there is an explanation which makes confusion on users' side:\r\n{quote}The {{-pr}} flag will only repair the “primary” ranges on a node, so you can repair your entire cluster by running {{nodetool repair -pr}} on each node in a single datacenter.\r\n{quote}\r\nThis made confusion that ""running nodetool repair -pr on each node in a single DC"" was sufficient to do a full repair on a multi-DC cluster.\r\n\r\nThe correct way to use the primary range repair should be\r\n{quote}if you are using “nodetool repair -pr” you must run it on *EVERY* node in *EVERY* data center, no skipping allowed.\r\n{quote}\r\nPlease make a doc fix. Thanks.",cassandra
5,Tiny documentation fix in Architecture/Overview,13304005,Normal,Fixed,11/May/20 04:33,20/Jun/23 19:24,"I noticed a small issue in the documentation, so I'd like to fix it.\r\n\r\nThis issue also exists to guide me to the proper way to make contributions to Cassandra, hopefully larger ones in the future, so please forgive the trivialness of this fix.",cassandra
6,Please update Apache Cassandra Repo link in install docs,13307023,Normal,Won't Fix,24/May/20 06:56,20/Jun/23 19:23,In the very first section of Install Docs\r\n\r\nAdd the Apache Cassandra repository keys to the list of trusted keys on the server:\r\n\r\n \r\n$ curl https://www.apache.org/dist/cassandra/KEYS | sudo apt-key - [Not working]\r\n \r\nPlease change the address. I found this working:\r\n \r\ncurl https://downloads.apache.org/cassandra/KEYS | sudo apt-key add -\r\n \r\n,cassandra
...,...,...,...,...,...,...,...,...
161165,macOS update-verify tests are failing on 136.0b2,1947772,P1,FIXED,2025-02-12 15:02:41+00:00,2025-03-03 12:13:12+00:00,macOS update-verify tests are failing on 136.0b2 \nhttps://firefox-ci-tc.services.mozilla.com/tasks/W1g5wn3AT9iKu2xUFA5Sjw\n\n```\n[task 2025-02-11T23:33:09.091Z] Unpacking downloads/Thunderbird 99.0b2.dmg with dmg and hfsplus\n[task 2025-02-11T23:33:09.091Z] ../common/unpack.sh: line 39: dmg: command not found\n[task 2025-02-11T23:33:09.092Z] ../common/unpack.sh: line 40: hfsplus: command not found\n```\nWe need to add `linux64-libdmg` to release-update-verify for macOS to pull in dmg and hfsplus.,thunderbird
161187,"Compose Window search bar text ""Replace..."" can overwrite the ""Whole words"" option",1948427,P3,FIXED,2025-02-14 23:22:23+00:00,2025-02-28 11:09:30+00:00,"User Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:135.0) Gecko/20100101 Firefox/135.0\n\nSteps to reproduce:\n\nVersion is 128.7\n\nIn an open compose window open the search bar (Ctrl+F), search for a string that is present in the message. This adds a ""Replace..."" option in the search bar.\n\n\nActual results:\n\nIn a normal-sized window for my 2560x1440 display the text of ""Replace..."" will partially overwrite the ""Whole words"" option to its left (see screenshots). If I widen the window enough to contain all the options then the display is OK.\n\n\nExpected results:\n\nText should not be overwritten",thunderbird
161188,"Intermittent comm/mail/test/browser/folder-display/browser_folderPaneVisibility.js | /browser_folderPaneVisibility.js | uncaught exception - TypeError: can't access property ""size"", folderTree.selection is 